In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/workspace /content/

In [ ]:
!dir /content/workspace/checkpoints

cnn_baseline_best.pth  convnext_tiny_unfreeze-1_best.pth


# Pre-requisite

In [ ]:
!pip install kaggle -q
!pip install albumentations -q
!pip install timm -q

In [ ]:
!kaggle datasets download -d puneet6060/intel-image-classification


Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
100% 346M/346M [00:20<00:00, 17.9MB/s]



# Build folder structure and Unzip dataset

In [ ]:
!pwd

/content


```
/content/
├── dataset/
└── workspace/
    ├── checkpoints/
    ├── logs/
    └── outputs/
```

In [ ]:
!mkdir /content/dataset
!mkdir -p /content/workspace/{checkpoints,logs,outputs}

In [ ]:
!unzip -q intel-image-classification.zip -d /content/dataset/intel-image-classification

In [ ]:
!dir /content/dataset/intel-image-classification/

seg_pred  seg_test  seg_train


In [ ]:
import shutil

refined_dir = '/content/dataset/refined'
!mkdir -p {refined_dir}/seg_train/seg_train
!mkdir -p {refined_dir}/seg_test/seg_test
!cp -r /content/dataset/intel-image-classification/seg_train/seg_train/* {refined_dir}/seg_train/seg_train/
!cp -r /content/dataset/intel-image-classification/seg_test/seg_test/*   {refined_dir}/seg_test/seg_test/

In [ ]:
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# data_dir = '/content/dataset/intel-image-classification'
data_dir = '/content/dataset/refined'

class IntelDataset(Dataset):
    def __init__(self, split, transform=None, use_synthetic=False):
        self.transform = transform
        self.samples = []
        self.classes = sorted(os.listdir(os.path.join(data_dir, split, split)))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        split_path = os.path.join(data_dir, split, split)
        for cls in self.classes:
            cls_path = os.path.join(split_path, cls)
            if os.path.isdir(cls_path):
                for img_file in os.listdir(cls_path):
                    if not use_synthetic and img_file.startswith('synthetic_'):
                        continue
                    self.samples.append((os.path.join(cls_path, img_file), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = np.array(Image.open(img_path).convert('RGB'))
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

### Load ConvNext-Tiny model architecture

NOTE: require convnext_tiny_best to run below codes

In [ ]:
import timm
import torch

experiment_name = 'convnext_tiny_best'
ckpt_path = os.path.join('/content/workspace/checkpoints', f'{experiment_name}_best.pth')

# model
model = timm.create_model('convnext_tiny.fb_in22k_ft_in1k', pretrained=False, num_classes=6)
checkpoint = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict(checkpoint['model'])
model = model.to('cuda')

# Dataset refinement to correct noisy labels

In [ ]:
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# from seg_train
DATASET_MEAN = (0.4302, 0.4575, 0.4538)
DATASET_STD = (0.2694, 0.2679, 0.2983)

eval_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=DATASET_MEAN, std=DATASET_STD),
    ToTensorV2(),
])

In [ ]:
import torch
import matplotlib.pyplot as plt

def evaluate_full(model, loader, class_names):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()

    all_preds, all_labels, all_imgs, all_probs = [], [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            probs   = torch.softmax(outputs, dim=1)
            preds   = outputs.argmax(dim=1)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            all_imgs.append(imgs.cpu())
            all_probs.append(probs.cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_imgs   = torch.cat(all_imgs)
    all_probs  = torch.cat(all_probs).numpy()

    return all_preds, all_labels, all_imgs, all_probs

def find_suspicious(all_preds, all_labels, all_probs, all_imgs, class_names, conf_threshold=0.9):
    suspicious = []
    for i in range(len(all_preds)):
        if all_preds[i] != all_labels[i]:
            conf = all_probs[i, all_preds[i]]
            if conf >= conf_threshold:
                suspicious.append({
                    'idx'      : i,
                    'gt'       : class_names[all_labels[i]],
                    'pred'     : class_names[all_preds[i]],
                    'conf'     : conf,
                    'img'      : all_imgs[i],
                })
    # sort by confidence descending
    suspicious = sorted(suspicious, key=lambda x: x['conf'], reverse=True)
    print(f"Found {len(suspicious)} suspicious samples with conf >= {conf_threshold}")
    return suspicious

def visualize_suspicious_paged(suspicious, page_size=10):
    mean = torch.tensor(DATASET_MEAN).view(3, 1, 1)
    std  = torch.tensor(DATASET_STD).view(3, 1, 1)

    total_pages = (len(suspicious) + page_size - 1) // page_size

    for page in range(total_pages):
        batch = suspicious[page * page_size:(page + 1) * page_size]
        n     = len(batch)

        fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), dpi=150)
        if n == 1:
            axes = [axes]
        fig.suptitle(f'Suspicious Labels — Page {page+1}/{total_pages} '
                     f'({page*page_size+1}–{page*page_size+n} of {len(suspicious)})',
                     fontsize=12, fontweight='bold')

        for i, sample in enumerate(batch):
            img = (sample['img'] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
            axes[i].imshow(img)
            axes[i].axis('off')
            axes[i].set_title(
                f"idx: {sample['idx']}\n"
                f"GT:   {sample['gt']}\n"
                f"Pred: {sample['pred']}\n"
                f"Conf: {sample['conf']:.2f}",
                fontsize=8, color='red', pad=4
            )

        plt.tight_layout()
        plt.savefig(os.path.join('/content/workspace/outputs',
                    f'suspicious_page{page+1:02d}.png'), bbox_inches='tight')
        plt.show()

def apply_refinement(dataset, corrections, class_names, refined_split_path):
    class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

    for idx, correct_label in corrections.items():
        img_path, old_label = dataset.samples[idx]
        old_cls = class_names[old_label]
        new_cls = correct_label

        # build refined path
        filename     = os.path.basename(img_path)
        old_dst_path = os.path.join(refined_split_path, old_cls, filename)
        new_dst_path = os.path.join(refined_split_path, new_cls, filename)

        if os.path.exists(old_dst_path):
            os.makedirs(os.path.join(refined_split_path, new_cls), exist_ok=True)
            shutil.move(old_dst_path, new_dst_path)
            print(f"  [{idx}] {old_cls} → {new_cls}: {filename}")

    print(f"\nApplied {len(corrections)} corrections")

Found 52 suspicious samples with conf >= 0.9


## Refine training set

In [ ]:
train_dataset = IntelDataset('seg_train', transform=eval_transform, use_synthetic=False)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

all_preds, all_labels, all_imgs, all_probs = evaluate_full(model, train_loader, train_dataset.classes)
suspicious = find_suspicious(all_preds, all_labels, all_probs, all_imgs, train_dataset.classes)
visualize_suspicious_paged(suspicious, page_size=4)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# manaully reviewed and filled out train_corrections
train_corrections = {}
train_corrections[5995] = 'mountain'
train_corrections[1304] = 'street'
train_corrections[612]  = 'street'
train_corrections[2566] = 'mountain'
train_corrections[4889] = 'mountain'
train_corrections[8192] = 'glacier'
train_corrections[5562] = 'mountain'
train_corrections[6157] = 'mountain'
train_corrections[7097] = 'glacier'
train_corrections[3697] = 'mountain'
train_corrections[4721] = 'mountain'
train_corrections[1531] = 'street'
train_corrections[8279] = 'glacier'
train_corrections[8268] = 'glacier'
train_corrections[4916] = 'mountain'
train_corrections[4478] = 'mountain'
train_corrections[5008] = 'mountain'
train_corrections[1777] = 'street'
train_corrections[4963] = 'mountain'
train_corrections[8431] = 'glacier'
train_corrections[11242] = 'mountain'
train_corrections[4856] = 'mountain'
train_corrections[5269] = 'mountain'
train_corrections[7763] = 'glacier'
train_corrections[5824] = 'mountain'
train_corrections[6850] = 'mountain'
train_corrections[8832] = 'glacier'
train_corrections[1453] = 'street'
train_corrections[5669] = 'mountain'
train_corrections[983]  = 'street'
train_corrections[7590] = 'glacier'
train_corrections[1808] = 'street'
train_corrections[6104] = 'mountain'
train_corrections[3002] = 'mountain'
train_corrections[14]   = 'street'
train_corrections[6413] = 'mountain'
train_corrections[2071] = 'street'
train_corrections[5702] = 'sea'
train_corrections[6673] = 'mountain'
train_corrections[7106] = 'glacier'
train_corrections[1488] = 'street'
train_corrections[5538] = 'mountain'
train_corrections[5777] = 'mountain'
train_corrections[9948] = 'mountain'
train_corrections[9668] = 'mountain'

train_corrections = {k: v for k, v in train_corrections.items()}
print(f"Total corrections in training set: {len(train_corrections)}")

Total corrections in training set: 45


In [ ]:
apply_refinement(train_dataset, train_corrections,
                 train_dataset.classes,
                 f'{refined_dir}/seg_train/seg_train')


Applied 45 corrections


## Refine test set

In [ ]:
test_dataset = IntelDataset('seg_test', transform=eval_transform, use_synthetic=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

all_preds, all_labels, all_imgs, all_probs = evaluate_full(model, test_loader, test_dataset.classes)
suspicious = find_suspicious(all_preds, all_labels, all_probs, all_imgs, test_dataset.classes)
visualize_suspicious_paged(suspicious, page_size=4)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# manaully reviewed and filled out test_corrections
test_corrections = {}
test_corrections[1775] = 'glacier'
test_corrections[1135] = 'mountain'
test_corrections[1394] = 'sea'
test_corrections[190]  = 'street'
test_corrections[1861] = 'glacier'
test_corrections[2830] = 'buildings'
test_corrections[2838] = 'buildings'
test_corrections[1578] = 'glacier'
test_corrections[1678] = 'glacier'
test_corrections[1122] = 'mountain'
test_corrections[1443] = 'mountain'
test_corrections[1843] = 'glacier'
test_corrections[335]  = 'street'
test_corrections[2714] = 'buildings'
test_corrections[1116] = 'sea'
test_corrections[1686] = 'glacier'
test_corrections[2519] = 'buildings'
test_corrections[1307] = 'mountain'
test_corrections[1291] = 'mountain'
test_corrections[200]  = 'street'
test_corrections[1740] = 'glacier'
test_corrections[1348] = 'mountain'
test_corrections[1059] = 'mountain'
test_corrections[1040] = 'mountain'
test_corrections[1531] = 'glacier'
test_corrections[2905] = 'buildings'
test_corrections[913]  = 'mountain'
test_corrections[2102] = 'mountain'
test_corrections[997]  = 'sea'
test_corrections[955]  = 'mountain'

print(f"Total corrections in test set: {len(test_corrections)}")

Total corrections in test set: 30


In [ ]:
apply_refinement(test_dataset, test_corrections,
                 test_dataset.classes,
                 f'{refined_dir}/seg_test/seg_test')

  [1775] mountain → glacier: 20093.jpg
  [1135] glacier → mountain: 23169.jpg
  [1394] glacier → sea: 23450.jpg
  [190] buildings → street: 20846.jpg
  [1861] mountain → glacier: 20160.jpg
  [2830] street → buildings: 20286.jpg
  [2838] street → buildings: 20683.jpg
  [1578] mountain → glacier: 21592.jpg
  [1678] mountain → glacier: 21239.jpg
  [1122] glacier → mountain: 24176.jpg
  [1443] glacier → mountain: 20317.jpg
  [1843] mountain → glacier: 21368.jpg
  [335] buildings → street: 22031.jpg
  [2714] street → buildings: 21694.jpg
  [1116] glacier → sea: 23027.jpg
  [1686] mountain → glacier: 23838.jpg
  [2519] street → buildings: 22351.jpg
  [1307] glacier → mountain: 21657.jpg
  [1291] glacier → mountain: 24279.jpg
  [200] buildings → street: 20246.jpg
  [1740] mountain → glacier: 20857.jpg
  [1348] glacier → mountain: 20421.jpg
  [1059] glacier → mountain: 22890.jpg
  [1040] glacier → mountain: 21452.jpg
  [1531] mountain → glacier: 22124.jpg
  [2905] street → buildings: 22370.jpg

In [ ]:
import json

# save
def save_corrections(corrections, split, log_dir='/content/workspace/logs'):
    path = os.path.join(log_dir, f'corrections_{split}.json')
    with open(path, 'w') as f:
        json.dump(corrections, f, indent=2)
    print(f"Saved {len(corrections)} corrections → {path}")

# load
def load_corrections(split, log_dir='/content/workspace/logs'):
    path = os.path.join(log_dir, f'corrections_{split}.json')
    with open(path, 'r') as f:
        corrections = json.load(f)
    corrections = {int(k): v for k, v in corrections.items()}
    print(f"Loaded {len(corrections)} corrections from {path}")
    return corrections

# usage
save_corrections(train_corrections, 'seg_train')
save_corrections(test_corrections,  'seg_test')

# train_corrections = load_corrections('seg_train')
# test_corrections  = load_corrections('seg_test')

Saved 45 corrections → /content/workspace/logs/corrections_seg_train.json
Saved 30 corrections → /content/workspace/logs/corrections_seg_test.json


In [ ]:
!cp /content/workspace/logs/corrections_seg_train.json /content/drive/MyDrive/workspace/
!cp /content/workspace/logs/corrections_seg_test.json  /content/drive/MyDrive/workspace/

In [ ]:
!cp /content/workspace/outputs/suspicious_page*.png /content/drive/MyDrive/workspace/